In [ ]:
import numpy as np
import gstools as gs
import matplotlib.pyplot as plt
from scipy.ndimage import convolve
from ca_tools import get_neighbours, decide_new_state, run_ca, summarize_stability



x=y=z = range(100)
model_1 = gs.Gaussian(dim=3, var=1, len_scale=[2, 2, 2], angles=[0, 0, 0])

model_2 = gs.Gaussian(dim=3, var=1, len_scale=[5, 5, 5], angles=[0, 0, 0])

model_3 = gs.Gaussian(dim=3, var=1, len_scale=[10, 10, 10], angles=[0, 0, 0])

srf_1 = gs.SRF(model_1)
srf_2 = gs.SRF(model_2)
srf_3 = gs.SRF(model_3)

srf_1((x,y,z), mesh_type='structured' , seed=0)
srf_2((x,y,z), mesh_type='structured' , seed=1)
srf_3((x,y,z), mesh_type='structured' , seed=2)

mesh_1 = srf_1.to_pyvista() 
mesh_2 = srf_2.to_pyvista() 
mesh_3 = srf_3.to_pyvista() 

mesh_1.contour(isosurfaces=50).plot()
mesh_2.contour(isosurfaces=50).plot()
mesh_3.contour(isosurfaces=50).plot()


In [ ]:
import numpy as np
import gstools as gs
import matplotlib.pyplot as plt
from scipy.ndimage import convolve



x=y = range(100)
model_1_2d = gs.Gaussian(dim=2, var=1, len_scale=[2, 2], angles=[0, 0])

model_2_2d = gs.Gaussian(dim=2, var=1, len_scale=[1, 5], angles=[np.pi/4, np.pi/4])



srf_1_2d = gs.SRF(model_1_2d)
srf_2_2d = gs.SRF(model_2_2d)


srf_1_2d((x,y), mesh_type='structured' , seed=0)
srf_2_2d((x,y), mesh_type='structured' , seed=1)


mesh_1_2d = srf_1_2d.to_pyvista() 
mesh_2_2d = srf_2_2d.to_pyvista() 
 

mesh_1_2d.contour(isosurfaces=50).plot()
mesh_2_2d.contour(isosurfaces=50).plot()

field_1_2d = srf_1_2d.field
field_2_2d = srf_2_2d.field




In [ ]:
import numpy as np

lithotype_map = np.zeros((100, 100))

cut_1 = 0.0
cut_2 = 0.5

lithotype_map[(field_1_2d >= cut_1) & (field_2_2d < cut_2)] = 1

lithotype_map[(field_1_2d >= cut_1) & (field_2_2d >= cut_2)] = 2

plt.imshow(lithotype_map, cmap='viridis')


total = lithotype_map.size
print(f"Rock type 0: {(lithotype_map == 0).sum() / total * 100:.1f}%")
print(f"Rock type 1: {(lithotype_map == 1).sum() / total * 100:.1f}%")
print(f"Rock type 2: {(lithotype_map == 2).sum() / total * 100:.1f}%")


In [ ]:
checkpoints = [10, 100]

for generation in range(100):
    new_grid = lithotype_map.copy()
    for i in range(1, lithotype_map.shape[0]-1):
        for j in range(1, lithotype_map.shape[1]-1):
            neighbours = [lithotype_map[i-1, j], lithotype_map[i+1, j], lithotype_map[i, j-1], lithotype_map[i, j+1]]
            counts = np.bincount(neighbours, minlength=3)
            if counts.max() > 1:
                new_grid[i, j] = counts.argmax()
    lithotype_map = new_grid
    
    if generation+1 in checkpoints:
        total = lithotype_map.size
        print(f"Generation {generation+1}:")
        print(f"  Rock type 0: {(lithotype_map == 0).sum() / total * 100:.1f}%")
        print(f"  Rock type 1: {(lithotype_map == 1).sum() / total * 100:.1f}%")
        print(f"  Rock type 2: {(lithotype_map == 2).sum() / total * 100:.1f}%")

plt.imshow(lithotype_map, cmap='viridis')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


rule_diagram = np.zeros((100, 100))

rule_diagram[50:100, 0:50] = 1

rule_diagram[50:100, 50:100] = 2

plt.imshow(rule_diagram, origin='lower', cmap='viridis')
plt.show()

Project: Virtual Prototyping of a Deep Geothermal Aquifer (gemini special)
Background & Objective:
To maximize energy extraction and prevent wasted drilling resources, we need to mathematically model the fluid dynamics of a subsurface geothermal reservoir. Before we can run dynamic fluid simulations, we must first construct the static geological environment. This notebook uses Plurigaussian simulation to generate a highly accurate 2D cross-section of the subterranean rock layers based on limited core sample data.

Geological Targets:
Physical testing indicates the reservoir is composed of three distinct lithotypes. Our mathematical model must strictly enforce these real-world volumetric proportions:

Phase 0 (Impermeable Granite Matrix): 65%. This dense rock forms the primary boundary structure and prevents fluid escape.

Phase 1 (Porous Sandstone Channels): 25%. The open, sponge-like network that serves as the main conduit for heated fluids.

Phase 2 (Dense Quartz Inclusions): 10%. Solid, crystalline blocks scattered within the channels that force the fluid to divert and branch.

In [ ]:
import numpy as np
import gstools as gs
import pyvista as pv
import matplotlib.pyplot as plt
from scipy.stats import norm

granite_percent = 0.65
sandstone_percent = 0.25
quartz_percent = 0.10

cut_1 = norm.ppf(granite_percent)
cut_2 = norm.ppf(sandstone_percent/(quartz_percent + sandstone_percent))

grid = range(100)

model_a = gs.Gaussian(dim=3, var=1.0, len_scale=[80, 20, 10])
srf_a = gs.SRF(model_a, seed=1)
field_a = srf_a.structured([grid, grid, grid])

model_b = gs.Gaussian(dim=3, var=1.0, len_scale=[60, 60, 30])
srf_b = gs.SRF(model_b, seed=2)
field_b = srf_b.structured([grid, grid, grid])

reservoir_map = np.zeros((len(grid), len(grid), len(grid)), dtype=np.int32)
reservoir_map[(field_a >= cut_1) & (field_b < cut_2)] = 1
reservoir_map[(field_a >= cut_1) & (field_b >= cut_2)] = 2

print("Original:", (reservoir_map == 0).sum(), (reservoir_map == 1).sum(), (reservoir_map == 2).sum())
original_map = reservoir_map.copy()

In [ ]:
for generation in range(10):
    new_grid = reservoir_map.copy()
    for i in range(1, reservoir_map.shape[0]-1):
        for j in range(1, reservoir_map.shape[1]-1):
            for k in range(1, reservoir_map.shape[2]-1):
                neighbours = [
                    reservoir_map[i-1, j, k],
                    reservoir_map[i+1, j, k],
                    reservoir_map[i, j-1, k],
                    reservoir_map[i, j+1, k],
                    reservoir_map[i, j, k-1],
                    reservoir_map[i, j, k+1]]
                counts = np.bincount(neighbours, minlength=3)
                if counts.max() > 1:
                    new_grid[i, j, k] = counts.argmax()
    reservoir_map = new_grid
    

print("After:", (reservoir_map == 0).sum(), (reservoir_map == 1).sum(), (reservoir_map == 2).sum())


In [ ]:
depth_slice = 50

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original_map[:, :, depth_slice], cmap='copper')
axes[0].set_title(f'Original - slice at depth {depth_slice}')
axes[1].imshow(reservoir_map[:, :, depth_slice], cmap='copper')
axes[1].set_title(f'After CA - slice at depth {depth_slice}')
plt.show()

# pre ca
grid_pv = pv.ImageData(dimensions=(len(grid)+1, len(grid)+1, len(grid)+1))
grid_pv.cell_data['lithotype'] = original_map.flatten(order='F')
plotter = pv.Plotter()
plotter.add_text('Original')
plotter.add_mesh(grid_pv, scalars='lithotype', cmap='copper', smooth_shading=True)
plotter.show()

# post ca 
grid_pv = pv.ImageData(dimensions=(len(grid)+1, len(grid)+1, len(grid)+1))
grid_pv.cell_data['lithotype'] = reservoir_map.flatten(order='F')
plotter = pv.Plotter()
plotter.add_text('After CA')
plotter.add_mesh(grid_pv, scalars='lithotype', cmap='copper', smooth_shading=True)
plotter.show()

In [ ]:
depth_slice = 50

lithotype_2d = np.zeros((100, 100), dtype=int)
lithotype_2d[(field_a[:, :, depth_slice] >= cut_1) & (field_b[:, :, depth_slice] < cut_2)] = 1
lithotype_2d[(field_a[:, :, depth_slice] >= cut_1) & (field_b[:, :, depth_slice] >= cut_2)] = 2

original_2d = lithotype_2d.copy()

total = lithotype_2d.size
print("Before:")
print(f"  Rock type 0: {(lithotype_2d == 0).sum() / total * 100:.1f}%")
print(f"  Rock type 1: {(lithotype_2d == 1).sum() / total * 100:.1f}%")
print(f"  Rock type 2: {(lithotype_2d == 2).sum() / total * 100:.1f}%")

result, snaps, history = run_ca(lithotype_2d.copy(), generations=10, checkpoints=[10], threshold=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original_2d, cmap='copper')
axes[0].set_title('Before CA')
axes[1].imshow(result, cmap='copper')
axes[1].set_title('After CA (our library)')
plt.show()

summarize_stability(history, window=5, tolerance=1.0)

In [ ]:
import numpy as np
import gstools as gs
import pyvista as pv
import matplotlib.pyplot as plt
from scipy.stats import norm

from ca_tools import get_neighbours, decide_new_state, run_ca, plot_ca_evolution, summarize_stability


rng = np.random.default_rng(42)

# start fresh, before any earlier CA evolution
cementation_map = original_2d.copy()

# scatter a handful of locked "seed crystals" of quartz (type 2) to grow from
locked_mask = np.zeros_like(cementation_map, dtype=bool)
seed_count = 8
seed_rows = rng.integers(0, cementation_map.shape[0], size=seed_count)
seed_cols = rng.integers(0, cementation_map.shape[1], size=seed_count)
cementation_map[seed_rows, seed_cols] = 2
locked_mask[seed_rows, seed_cols] = True

# quartz really doesn't like touching anything else (cost 3), granite/sandstone
# interact normally with each other (cost 1) - this pushes quartz to grow as
# compact, round blobs rather than spreading thin
interaction_matrix = np.array([
    [0, 1, 3],
    [1, 0, 3],
    [3, 3, 0]
])

result, snaps, history = run_ca(
    cementation_map, generations=60, checkpoints=[10, 20, 30, 40, 50, 60],
    rule='probabilistic', temperature=2, interaction_matrix=interaction_matrix,
    update_scheme='asynchronous', locked_mask=locked_mask,
    nucleation_rate=0.0005, rng=rng
)

plot_ca_evolution(snaps)

In [ ]:
total = result.size
print(f"Final - Rock type 0: {(result==0).sum()/total*100:.1f}%, "
      f"1: {(result==1).sum()/total*100:.1f}%, 2: {(result==2).sum()/total*100:.1f}%")
summarize_stability(history, window=10, tolerance=1.0)

In [ ]:
# fresh 2D fields with a directional "grain" to the rock
model_1 = gs.Gaussian(dim=2, var=1, len_scale=[40, 10], angles=[np.pi/4])
srf_1 = gs.SRF(model_1, seed=10)
field_1 = srf_1.structured([range(100), range(100)])

model_2 = gs.Gaussian(dim=2, var=1, len_scale=[20, 20])
srf_2 = gs.SRF(model_2, seed=11)
field_2 = srf_2.structured([range(100), range(100)])

cut_1 = norm.ppf(0.65)
cut_2 = norm.ppf(0.25 / (0.25 + 0.10))

vein_map = np.zeros((100, 100), dtype=int)
vein_map[(field_1 >= cut_1) & (field_2 < cut_2)] = 1
vein_map[(field_1 >= cut_1) & (field_2 >= cut_2)] = 2

# seed a short quartz vein along the NE-SW diagonal through the centre
locked_mask = np.zeros_like(vein_map, dtype=bool)
for offset in range(-3, 4):
    vein_map[50 + offset, 50 - offset] = 2
    locked_mask[50 + offset, 50 - offset] = True

# bias growth along NE/SW much more than the other directions
vein_weights = {
    (-1, 1): 4, (1, -1): 4,                          # NE / SW - strongly favoured
    (-1, -1): 1, (1, 1): 1,                          # NW / SE - discouraged
    (-1, 0): 2, (1, 0): 2, (0, -1): 2, (0, 1): 2,     # N/S/E/W - moderate
}

rng = np.random.default_rng(7)

result, snaps, history = run_ca(
    vein_map, generations=40, checkpoints=[10, 20, 30, 40],
    neighbourhood='moore', weights=vein_weights,
    rule='majority', threshold=4,
    update_scheme='asynchronous', locked_mask=locked_mask,
    nucleation_rate=0.0003,
    target_proportions={0: 0.65, 1: 0.25, 2: 0.10}, conservation_strength=0.4,
    rng=rng
)

plot_ca_evolution(snaps)

In [3]:
import numpy as np
import gstools as gs
import matplotlib.pyplot as plt
from scipy.stats import norm

from ca_tools import get_neighbours, decide_new_state, run_ca, plot_ca_evolution, summarize_stability

# fresh 2D fields with a directional "grain" to the rock
model_1 = gs.Gaussian(dim=2, var=1, len_scale=[40, 10], angles=[np.pi/4])
srf_1 = gs.SRF(model_1, seed=10)
field_1 = srf_1.structured([range(100), range(100)])

model_2 = gs.Gaussian(dim=2, var=1, len_scale=[20, 20])
srf_2 = gs.SRF(model_2, seed=11)
field_2 = srf_2.structured([range(100), range(100)])

cut_1 = norm.ppf(0.65)
cut_2 = norm.ppf(0.25 / (0.25 + 0.10))

vein_map = np.zeros((100, 100), dtype=int)
vein_map[(field_1 >= cut_1) & (field_2 < cut_2)] = 1
vein_map[(field_1 >= cut_1) & (field_2 >= cut_2)] = 2

# seed a short quartz vein along the NE-SW diagonal through the centre
locked_mask = np.zeros_like(vein_map, dtype=bool)
for offset in range(-3, 4):
    vein_map[50 + offset, 50 - offset] = 2
    locked_mask[50 + offset, 50 - offset] = True

# bias growth along NE/SW much more than the other directions
vein_weights = {
    (-1, 1): 4, (1, -1): 4,                          # NE / SW - strongly favoured
    (-1, -1): 1, (1, 1): 1,                          # NW / SE - discouraged
    (-1, 0): 2, (1, 0): 2, (0, -1): 2, (0, 1): 2,     # N/S/E/W - moderate
}

rng = np.random.default_rng(7)

result, snaps, history = run_ca(
    vein_map, generations=1000, checkpoints=range(0, 1001, 10),
    neighbourhood='moore', weights=vein_weights,
    rule='majority', threshold=4,
    update_scheme='asynchronous', locked_mask=locked_mask,
    nucleation_rate=0.0003,
    target_proportions={0: 0.65, 1: 0.25, 2: 0.10}, conservation_strength=0.4,
    rng=rng
)

plot_ca_evolution(snaps)

Generation 10:
  Rock type 0: 59.6%
  Rock type 1: 29.3%
  Rock type 2: 11.1%
Generation 20:
  Rock type 0: 69.5%
  Rock type 1: 22.4%
  Rock type 2: 8.1%
Generation 30:
  Rock type 0: 76.7%
  Rock type 1: 18.2%
  Rock type 2: 5.1%
Generation 40:
  Rock type 0: 83.8%
  Rock type 1: 13.1%
  Rock type 2: 3.1%
Generation 50:
  Rock type 0: 89.0%
  Rock type 1: 9.3%
  Rock type 2: 1.7%
Generation 60:
  Rock type 0: 93.0%
  Rock type 1: 6.5%
  Rock type 2: 0.5%
Generation 70:
  Rock type 0: 95.7%
  Rock type 1: 4.2%
  Rock type 2: 0.1%
Generation 80:
  Rock type 0: 98.2%
  Rock type 1: 1.8%
  Rock type 2: 0.1%
Generation 90:
  Rock type 0: 99.5%
  Rock type 1: 0.4%
  Rock type 2: 0.1%
Generation 100:
  Rock type 0: 99.9%
  Rock type 1: 0.0%
  Rock type 2: 0.1%
Generation 110:
  Rock type 0: 99.9%
  Rock type 1: 0.0%
  Rock type 2: 0.1%
Generation 120:
  Rock type 0: 99.9%
  Rock type 1: 0.0%
  Rock type 2: 0.1%
Generation 130:
  Rock type 0: 99.9%
  Rock type 1: 0.0%
  Rock type 2: 0.1%
Gen

In [4]:
total = result.size
print(f"Final - 0: {(result==0).sum()/total*100:.1f}%, 1: {(result==1).sum()/total*100:.1f}%, 2: {(result==2).sum()/total*100:.1f}%")
summarize_stability(history, window=10, tolerance=1.0)

Final - 0: 99.9%, 1: 0.0%, 2: 0.1%
Looks stable: max change over the last 10 generations was 0.02 percentage points.
